In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv

load_dotenv("../.env")

True

In [3]:
from configs.config import Config

from src.mongo_utils import (
    get_collection,
    count_documents,
)

from src.youtube_ingest import ingest_all_videos

from src.notebook_helpers import (
    display_ingestion_results,
    display_jobs_status,
    display_jobs_summary,
)

from src import ingestion_tracker as tracker

In [4]:
%store -r video_entries

print(f"Total video dalam daftar: {len(video_entries)}")

for entry in video_entries:
    print(f"- {entry['video_id']}")

Total video dalam daftar: 14
- _8H_-uNHKL4
- TcNSqpBPle4
- UEocTRWhtsE
- g7Et92hQ_yE
- 5btOJ_Wru2k
- tQKnVXSL7nc
- HcwZx39bP80
- UE2gk8vMgBQ
- LgaDFmaTpak
- u4zZ7gHRBaU
- uxDGtbE72jk
- 4yjXBLcjynU
- UTvTHgjI_QI
- BMcbB3SlU1M


In [5]:
print(video_entries)

[{'video_id': '_8H_-uNHKL4', 'video_url': 'https://www.youtube.com/watch?v=_8H_-uNHKL4&t=351s'}, {'video_id': 'TcNSqpBPle4', 'video_url': 'https://www.youtube.com/watch?v=TcNSqpBPle4'}, {'video_id': 'UEocTRWhtsE', 'video_url': 'https://www.youtube.com/watch?v=UEocTRWhtsE'}, {'video_id': 'g7Et92hQ_yE', 'video_url': 'https://www.youtube.com/watch?v=g7Et92hQ_yE&t=2237s'}, {'video_id': '5btOJ_Wru2k', 'video_url': 'https://www.youtube.com/watch?v=5btOJ_Wru2k'}, {'video_id': 'tQKnVXSL7nc', 'video_url': 'https://www.youtube.com/watch?v=tQKnVXSL7nc'}, {'video_id': 'HcwZx39bP80', 'video_url': 'https://www.youtube.com/watch?v=HcwZx39bP80'}, {'video_id': 'UE2gk8vMgBQ', 'video_url': 'https://www.youtube.com/watch?v=UE2gk8vMgBQ'}, {'video_id': 'LgaDFmaTpak', 'video_url': 'https://www.youtube.com/watch?v=LgaDFmaTpak'}, {'video_id': 'u4zZ7gHRBaU', 'video_url': 'https://www.youtube.com/watch?v=u4zZ7gHRBaU'}, {'video_id': 'uxDGtbE72jk', 'video_url': 'https://www.youtube.com/watch?v=uxDGtbE72jk'}, {'vid

In [6]:
collection_jobs = get_collection(
    uri=Config.mongo.URI,
    db_name=Config.mongo.DB_NAME,
    collection_name=Config.mongo.COLLECTION_JOBS,
)

collection_raw = get_collection(
    uri=Config.mongo.URI,
    db_name=Config.mongo.DB_NAME,
    collection_name=Config.mongo.COLLECTION_RAW,
)

print("[OK] MongoDB collections berhasil dimuat.")  

2026-05-13 23:58:13 | INFO | mongo_utils | Koneksi MongoDB berhasil.
[OK] MongoDB collections berhasil dimuat.


In [7]:
results = ingest_all_videos(
    video_entries=video_entries,
    api_key=Config.youtube.API_KEY,
    endpoint=Config.youtube.COMMENTS_ENDPOINT,
    base_url=Config.youtube.BASE_URL,
    collection_raw=collection_raw,
    collection_jobs=collection_jobs,
    target_count=Config.youtube.TARGET_COMMENT_COUNT,
    max_results_per_page=Config.youtube.MAX_RESULTS_PER_PAGE,
    flush_every=500,
    delay_seconds=0.5,
    inter_video_delay=2.0,
)

print("\n[OK] Ingestion selesai.")

2026-05-13 23:58:21 | INFO | youtube_ingest | [1/14] Memproses _8H_-uNHKL4...
2026-05-13 23:58:22 | INFO | youtube_ingest | Memulai ingestion: _8H_-uNHKL4 (#UGMMENJAWAB IJAZAH JOKO WIDODO)
2026-05-13 23:58:22 | INFO | youtube_ingest | Halaman 1: 100 komentar dikumpulkan (total: 100).
2026-05-13 23:58:23 | INFO | youtube_ingest | Halaman 2: 100 komentar dikumpulkan (total: 200).
2026-05-13 23:58:25 | INFO | youtube_ingest | Halaman 3: 100 komentar dikumpulkan (total: 300).
2026-05-13 23:58:26 | INFO | youtube_ingest | Halaman 4: 100 komentar dikumpulkan (total: 400).
2026-05-13 23:58:29 | INFO | youtube_ingest | Halaman 5: 100 komentar dikumpulkan (total: 500).
2026-05-13 23:58:30 | INFO | youtube_ingest | Halaman 6: 100 komentar dikumpulkan (total: 600).
2026-05-13 23:58:31 | INFO | youtube_ingest | Halaman 7: 100 komentar dikumpulkan (total: 700).
2026-05-13 23:58:33 | INFO | youtube_ingest | Halaman 8: 100 komentar dikumpulkan (total: 800).
2026-05-13 23:58:34 | INFO | youtube_ingest

In [8]:
from src.mongo_utils import count_documents

# Statistik total komentar di MongoDB
total_raw = count_documents(collection_raw)
print(f"\nTotal komentar tersimpan di MongoDB (semua video): {total_raw:,}")
display_ingestion_results(results)


Total komentar tersimpan di MongoDB (semua video): 14,107

  HASIL INGESTION BATCH
  _8H_-uNHKL4  completed                1,163 komentar
  TcNSqpBPle4  completed                1,250 komentar
  UEocTRWhtsE  completed                1,067 komentar
  g7Et92hQ_yE  completed                1,045 komentar
  5btOJ_Wru2k  completed                1,071 komentar
  tQKnVXSL7nc  completed                1,073 komentar
  HcwZx39bP80  completed                  958 komentar
  UE2gk8vMgBQ  completed                  856 komentar
  LgaDFmaTpak  completed                  999 komentar
  u4zZ7gHRBaU  completed                  871 komentar
  uxDGtbE72jk  completed                1,039 komentar
  4yjXBLcjynU  completed                1,021 komentar
  UTvTHgjI_QI  completed                  752 komentar
  BMcbB3SlU1M  completed                  942 komentar

